# 🧩 Python Multidimensional Arrays and Tensors — The Master Guide
*From Zero to Interview-Ready*

---

## Mental Model

A 1D array is a single shelf of boxes. A 2D array is a bookcase — shelves stacked on top of each other. A 3D array is a row of bookcases side by side — depth, rows, columns. Every operation — sum, reshape, slice — runs in a specific direction through that physical structure, and NumPy calls that direction the axis.

---

## Table of Contents

| # | Section | Link |
|---|---------|------|
| 1 | What Is an ndarray? The Visual Model | [#1](#1) |
| 2 | Creating / Setup | [#2](#2) |
| 3 | The Core API — shape, ndim, size, dtype, indexing | [#3](#3) |
| 4 | Decision Map — When To Use What | [#4](#4) |
| 5 | Pattern 1: NumPy ndarray — shape, ndim, size, dtype (LC 74, 240) | [#5](#5) |
| 6 | Pattern 2: Axis — which direction operations run (LC 48, 54) | [#6](#6) |
| 7 | Pattern 3: Reshape and Flatten — reshape, -1 inference, ravel (LC 48, 54) | [#7](#7) |
| 8 | Pattern 4: Broadcasting — stretch rules, elementwise ops (LC 238, 560) | [#8](#8) |
| 9 | Pattern 5: 3D Tensors — batch × rows × cols, PyTorch/NumPy parity | [#9](#9) |
| 10 | The Multidimensional Arrays Decision Map | [#10](#10) |
| 11 | Interview Cheat Sheet | [#11](#11) |

<a id='1'></a>

## 1. What Is an ndarray? The Visual Model

```
                    NUMPY NDARRAY — THE LAYERED BOX STRUCTURE

  1D — ONE SHELF (shape = (5,))
  ┌────┬────┬────┬────┬────┐
  │ 10 │ 20 │ 30 │ 40 │ 50 │
  └────┴────┴────┴────┴────┘
   [0]  [1]  [2]  [3]  [4]

  2D — BOOKCASE (shape = (3, 4))  ← 3 shelves, 4 boxes per shelf
                   axis=1 →→→→→→
           ┌────┬────┬────┬────┐
  axis=0   │  1 │  2 │  3 │  4 │  row 0
    ↓      ├────┼────┼────┼────┤
    ↓      │  5 │  6 │  7 │  8 │  row 1
    ↓      ├────┼────┼────┼────┤
           │  9 │ 10 │ 11 │ 12 │  row 2
           └────┴────┴────┴────┘
           col0 col1 col2 col3

  3D — ROW OF BOOKCASES (shape = (2, 3, 4))  ← 2 bookcases, 3 shelves, 4 boxes
       axis=0 selects the bookcase
       axis=1 selects the shelf (row) within a bookcase
       axis=2 selects the box (column) within a shelf

       bookcase 0               bookcase 1
       ┌────┬────┬────┬────┐   ┌────┬────┬────┬────┐
       │  1 │  2 │  3 │  4 │   │ 13 │ 14 │ 15 │ 16 │
       ├────┼────┼────┼────┤   ├────┼────┼────┼────┤
       │  5 │  6 │  7 │  8 │   │ 17 │ 18 │ 19 │ 20 │
       ├────┼────┼────┼────┤   ├────┼────┼────┼────┤
       │  9 │ 10 │ 11 │ 12 │   │ 21 │ 22 │ 23 │ 24 │
       └────┴────┴────┴────┘   └────┴────┴────┴────┘
         shape[1]=3 rows          shape[2]=4 cols

  AXIS DIRECTION RULE
  ─────────────────────────────────────────────────────
  np.sum(a, axis=0)  → sum DOWN   the rows → result shape (4,)   for (3,4) input
  np.sum(a, axis=1)  → sum ACROSS the cols → result shape (3,)   for (3,4) input
  axis collapses the dimension it runs along — think of it being "eaten"

  WHY THIS MATTERS
  ─────────────────────────────────────────────────────
  Every aggregation (sum, mean, max) collapses one axis.
  np.stack adds a NEW axis. np.concatenate extends an EXISTING axis.
  Knowing the axis is what prevents off-by-one shape bugs in ML pipelines.
  In PyTorch: axis= is spelled dim= but the rule is IDENTICAL.
```

<a id='2'></a>

## 2. Creating / Setup

In [ ]:
import numpy as np

# ── 1D arrays ───────────────────────────────────────────────
a1 = np.array([10, 20, 30, 40, 50])          # from Python list
a2 = np.zeros(5)                              # five 0.0s
a3 = np.ones(5, dtype=int)                    # five 1s, integer type
a4 = np.arange(0, 10, 2)                      # [0,2,4,6,8] — like range()
a5 = np.linspace(0, 1, 5)                     # 5 evenly spaced floats 0→1

# ── 2D arrays ───────────────────────────────────────────────
b1 = np.array([[1,2,3],[4,5,6]])              # shape (2,3) from nested lists
b2 = np.zeros((3, 4))                         # 3x4 of 0.0
b3 = np.ones((2, 2), dtype=int)               # 2x2 of 1s
b4 = np.eye(3)                                # 3x3 identity matrix
b5 = np.arange(12).reshape(3, 4)             # 0..11 reshaped to 3 rows x 4 cols

# ── 3D arrays ───────────────────────────────────────────────
c1 = np.zeros((2, 3, 4))                      # 2 bookcases, 3 shelves, 4 boxes
c2 = np.arange(24).reshape(2, 3, 4)          # 0..23 as a 3D tensor

print(f"a1          shape={a1.shape}  ndim={a1.ndim}  size={a1.size}")
print(f"b5          shape={b5.shape}  ndim={b5.ndim}  size={b5.size}")
print(f"c2          shape={c2.shape}  ndim={c2.ndim}  size={c2.size}")
print(f"b5 dtype    : {b5.dtype}")
print(f"b5:\n{b5}")
print(f"c2:\n{c2}")

# Simplicity and clarity is Gold

<a id='3'></a>

## 3. The Core API — shape, ndim, size, dtype, indexing

```
ATTRIBUTE / OPERATION     RETURNS / EFFECT          EXAMPLE
─────────────────────────────────────────────────────────────────────
a.shape                   tuple of dim sizes         (3, 4)
a.ndim                    number of dimensions       2
a.size                    total number of elements   12
a.dtype                   element data type          int64, float64
a.T                       transposed view            shape (4,3) from (3,4)
a[i, j]                   element at row i, col j    O(1)
a[i]                      full row i (1D view)        O(1) — no copy
a[:, j]                   full column j (1D view)     O(1) — no copy
a[i:k, j:l]               sub-matrix slice            O(k*l) — view when possible
np.sum(a, axis=0)          sum down rows → shape (n,)
np.sum(a, axis=1)          sum across cols → shape (m,)
np.mean(a, axis=0)         mean down rows
a.reshape(r, c)            new shape, same data       r*c must equal a.size
a.reshape(-1)              flatten to 1D              -1 = infer this dim
a.ravel()                  flatten to 1D view         no copy when possible
a.flatten()                flatten to 1D copy         always a copy
np.dot(a, b)               matrix multiply            (m,k) @ (k,n) → (m,n)
a @ b                      same as np.dot for 2D
np.stack([a,b], axis=0)    new axis at position 0     two (3,4) → (2,3,4)
np.concatenate([a,b], 0)   join along existing axis   two (3,4) → (6,4)
a + b                      elementwise — broadcasts
a * b                      elementwise — broadcasts
a.astype(float)            cast to new dtype — copy

THINGS YOU DO NOT DO
─────────────────────────────────────────────────────────────────────
❌  a.shape = (n, m)      works but confusing — use reshape() instead
❌  a[0] for 2D            gives a row view — mutation propagates back to a
❌  np.sum(a) with no axis  collapses EVERYTHING to a scalar — often unintended
❌  reshape without checking r*c == a.size  → raises ValueError
❌  assume reshape makes a copy — it usually makes a view (same memory)
```

In [ ]:
import numpy as np

a = np.arange(12).reshape(3, 4)
print(f"a:\n{a}")
print(f"shape={a.shape}  ndim={a.ndim}  size={a.size}  dtype={a.dtype}")

# indexing
print(f"a[1,2]    = {a[1,2]}")          # element at row 1, col 2 = 6
print(f"a[0]      = {a[0]}")            # full row 0 = [0 1 2 3]
print(f"a[:,1]    = {a[:,1]}")          # full col 1 = [1 5 9]
print(f"a[0:2,1:3]:\n{a[0:2,1:3]}")    # sub-matrix rows 0-1, cols 1-2

# axis operations
print(f"sum axis=0: {np.sum(a, axis=0)}")  # sum down rows  -> shape (4,)
print(f"sum axis=1: {np.sum(a, axis=1)}")  # sum across cols -> shape (3,)
print(f"mean axis=0: {np.mean(a, axis=0)}")

# transpose
print(f"a.T shape: {a.T.shape}")
print(f"a.T:\n{a.T}")

# reshape
flat   = a.reshape(-1)          # infer -> 1D of 12 elements
col    = a.reshape(-1, 1)       # infer rows, force 1 column -> shape (12,1)
back   = flat.reshape(3, 4)     # back to original shape
print(f"reshape(-1) shape: {flat.shape}")
print(f"reshape(-1,1) shape: {col.shape}")
print(f"reshape(3,4) shape: {back.shape}")

# Simplicity and clarity is Gold

<a id='4'></a>

## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                        WHAT TO DO
──────────────────────────────────────────────────────────────────────────
need to know structure of array              a.shape, a.ndim, a.size
sum/mean per row (collapse columns)          np.sum(a, axis=1)
sum/mean per column (collapse rows)          np.sum(a, axis=0)
rotate matrix 90 degrees clockwise           np.rot90(a, k=-1) or transpose+reverse
search sorted 2D matrix                      a.ravel() + binary search, or // and %
flatten to 1D for scanning                   a.ravel() (view) or a.flatten() (copy)
change shape without changing data           a.reshape(r, c) — check r*c == a.size
add a scalar to every element                a + scalar  — broadcasting
add a (1,4) row vector to a (3,4) matrix     a + row_vec — broadcasting stretches row
batch of matrices (3D)                       shape (batch, rows, cols) = (B,H,W)
need matrix multiply                         a @ b or np.dot(a, b)
stack two 2D arrays into 3D                  np.stack([a, b], axis=0)
```

<a id='5'></a>

## 5. 🧩 Pattern 1: NumPy ndarray — shape, ndim, size, dtype — LC 74, 240

```
PROBLEM:
  LC 74 — Search a 2D Matrix  /  LC 240 — Search a 2D Matrix II
  Both require understanding the matrix structure before choosing a strategy.
  LC 74: strictly sorted rows AND last of row i < first of row i+1 → treat as 1D.
  LC 240: each row sorted, each col sorted → staircase search from top-right.

NDARRAY FUNDAMENTALS:
  shape = (m, n) → m rows, n columns
  ndim  = 2      → two axes (axis=0 rows, axis=1 cols)
  size  = m*n    → total elements
  dtype          → element type — int64, float64, bool, etc.

  Reading shape[0] and shape[1] is how you get m and n safely.
  a.ravel() flattens to 1D for LC 74 binary search.
  Direct indexing a[row, col] is how you verify in LC 240 staircase.

SLOW MOTION TRACE — LC 240 staircase on:
  matrix = [[1,4,7,11],[2,5,8,12],[3,6,9,16],[10,13,14,17]]  target=5

  start at top-right: row=0, col=3 → val=11
    11 > 5  → col -= 1  → col=2, val=7
    7  > 5  → col -= 1  → col=1, val=4
    4  < 5  → row += 1  → row=1, val=matrix[1][1]=5
    5 == 5  → FOUND  ✓

KEY INSIGHT:
  Top-right corner is a decision point: larger than anything in its column above,
  smaller than anything in its row to the right.
  > target → eliminate this column (move left)
  < target → eliminate this row (move down)

TIME / SPACE (LC 240):
  Time:  O(m+n) — at most m+n steps (move left or down, never both)
  Space: O(1)   — only row/col pointers
```

In [ ]:
import numpy as np

def search_matrix_ii(matrix: list, target: int) -> bool:
    """
    LC 240 -- Search a 2D Matrix II
    Approach: staircase search from top-right corner.
    Args:
        matrix (list[list[int]]): m x n, each row sorted, each col sorted.
        target (int): value to find.
    Returns:
        bool: True if target found.
    Time:  O(m+n) -- at most m+n steps, one axis moves per step
    Space: O(1)   -- two index variables
    """
    if not matrix or not matrix[0]:
        return False

    m = len(matrix)         # number of rows -- matrix.shape[0] in NumPy
    n = len(matrix[0])      # number of cols -- matrix.shape[1] in NumPy

    row = 0                 # start at top row
    col = n - 1             # start at rightmost column

    while row < m and col >= 0:
        val = matrix[row][col]

        # slow motion on [[1,4,7,11],[2,5,8,12],[3,6,9,16],[10,13,14,17]], target=5:
        # row=0 col=3 val=11  11>5  -> col=2
        # row=0 col=2 val=7   7>5   -> col=1
        # row=0 col=1 val=4   4<5   -> row=1
        # row=1 col=1 val=5   found!

        if val == target:
            return True
        elif val > target:
            col -= 1    # everything in this column below is also too big -- eliminate col
        else:
            row += 1    # everything in this row to the left is also too small -- eliminate row

    return False


def show_ndarray_anatomy(matrix_list):
    """Convert a list-of-lists to ndarray and display all key attributes."""
    a = np.array(matrix_list)
    print(f"array:\n{a}")
    print(f"shape  = {a.shape}    (rows={a.shape[0]}, cols={a.shape[1]})")
    print(f"ndim   = {a.ndim}")
    print(f"size   = {a.size}   (total elements)")
    print(f"dtype  = {a.dtype}")
    print(f"ravel  = {a.ravel()}")   # flat view -- use for 1D scan


def test_harness(fn):
    tests = [
        ([[1,4,7,11],[2,5,8,12],[3,6,9,16],[10,13,14,17]], 5,   True),
        ([[1,4,7,11],[2,5,8,12],[3,6,9,16],[10,13,14,17]], 20,  False),
        ([[1,4,7,11],[2,5,8,12],[3,6,9,16],[10,13,14,17]], 1,   True),
        ([[1,4,7,11],[2,5,8,12],[3,6,9,16],[10,13,14,17]], 17,  True),
        ([[1]],                                             1,   True),
        ([[1]],                                             2,   False),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(search_matrix_ii)
show_ndarray_anatomy([[1,4,7,11],[2,5,8,12],[3,6,9,16],[10,13,14,17]])

print("search_matrix_ii defined.")

<a id='6'></a>

## 6. 🧩 Pattern 2: Axis — which direction operations run — LC 48, 54

```
PROBLEM:
  LC 48 — Rotate Image (90 degrees clockwise, in-place)
  LC 54 — Spiral Matrix (collect elements in spiral order)

AXIS CONCEPT:
  axis=0 runs DOWN  — through the rows — result has one value per COLUMN
  axis=1 runs ACROSS — through the columns — result has one value per ROW

  Think of the axis number as the index that GETS CONSUMED.
  np.sum(a, axis=0) → axis 0 (rows) gets consumed → result shape loses axis 0
  np.sum(a, axis=1) → axis 1 (cols) gets consumed → result shape loses axis 1

  (3,4) array:
    sum axis=0 → shape (4,)   one sum per column (rows collapsed)
    sum axis=1 → shape (3,)   one sum per row    (cols collapsed)

AXIS AND ROTATION (LC 48):
  np.rot90(a, k=1)   → 90 degrees counter-clockwise
  np.rot90(a, k=-1)  → 90 degrees clockwise (k=-1 same as k=3)
  Manual: a.T then np.fliplr(a.T) → clockwise

  Transpose a.T:         flip left-right np.fliplr:
  1 2 3    →  1 4 7      →  7 4 1
  4 5 6        2 5 8         8 5 2
  7 8 9        3 6 9         9 6 3

SLOW MOTION TRACE — axis=0 sum on:
  a = [[1,2,3],        np.sum(a, axis=0):
       [4,5,6]]          col0: 1+4=5
                          col1: 2+5=7
                          col2: 3+6=9
                         result: [5, 7, 9]  shape=(3,)

  np.sum(a, axis=1):
    row0: 1+2+3=6
    row1: 4+5+6=15
    result: [6, 15]  shape=(2,)

KEY INSIGHT:
  "axis collapses the dimension it names."
  If you cannot remember which axis does what, test with a (2,3) shape.
  axis=0 result is shape (3,) — axis 0 (size 2) was consumed.
  axis=1 result is shape (2,) — axis 1 (size 3) was consumed.

TIME / SPACE (LC 48 via NumPy):
  Time:  O(n^2) — visit every element
  Space: O(1)  — rot90 and fliplr operate on views when possible
```

In [ ]:
import numpy as np

def rotate_numpy(matrix: list) -> list:
    """
    LC 48 -- Rotate Image (90 degrees clockwise) using NumPy axis operations.
    Approach: transpose then flip left-right (same as transpose + row-reverse).
    Args:
        matrix (list[list[int]]): n x n matrix.
    Returns:
        list: new rotated matrix as list-of-lists.
    Time:  O(n^2) -- full matrix traversal
    Space: O(n^2) -- NumPy creates new array (problem allows this for demonstration)
    """
    a = np.array(matrix)

    # transpose: rows become columns
    # slow motion on [[1,2,3],[4,5,6],[7,8,9]]:
    # after .T:  [[1,4,7],[2,5,8],[3,6,9]]
    transposed = a.T

    # flip left-right: each row reverses
    # after fliplr: [[7,4,1],[8,5,2],[9,6,3]]  <- 90 degree clockwise result
    rotated = np.fliplr(transposed)

    return rotated.tolist()


# -- AXIS DEMO ---------------------------------------------------------
a = np.array([[1, 2, 3],
              [4, 5, 6]])

print(f"a:\n{a}  shape={a.shape}")

# axis=0 -- runs down rows -- result has one value per column
print(f"np.sum(a, axis=0)  = {np.sum(a, axis=0)}  shape={np.sum(a, axis=0).shape}")
print(f"np.mean(a, axis=0) = {np.mean(a, axis=0)}")

# axis=1 -- runs across columns -- result has one value per row
print(f"np.sum(a, axis=1)  = {np.sum(a, axis=1)}  shape={np.sum(a, axis=1).shape}")
print(f"np.mean(a, axis=1) = {np.mean(a, axis=1)}")

# no axis -- collapses everything
print(f"np.sum(a)          = {np.sum(a)}  (scalar)")

# rotation demo
m = [[1,2,3],[4,5,6],[7,8,9]]
print(f"original:\n{np.array(m)}")
print(f"rot90 k=1 (CCW):\n{np.rot90(np.array(m), k=1)}")
print(f"rot90 k=-1 (CW):\n{np.rot90(np.array(m), k=-1)}")
print(f"rotate_numpy result:\n{np.array(rotate_numpy(m))}")


def test_harness(fn):
    tests = [
        ([[1,2,3],[4,5,6],[7,8,9]],                       [[7,4,1],[8,5,2],[9,6,3]]),
        ([[5,1,9,11],[2,4,8,10],[13,3,6,7],[15,14,12,16]],[[15,13,2,5],[14,3,4,1],[12,6,8,9],[16,7,10,11]]),
        ([[1]],                                            [[1]]),
        ([[1,2],[3,4]],                                    [[3,1],[4,2]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(rotate_numpy)

print("rotate_numpy defined.")

<a id='7'></a>

## 7. 🧩 Pattern 3: Reshape and Flatten — reshape, -1 inference, ravel — LC 48, 54

```
PROBLEM:
  LC 48 — Rotate Image: working in 2D vs treating as flattened 1D
  LC 54 — Spiral Matrix: collect elements — output is 1D regardless of input shape

RESHAPE RULES:
  a.reshape(r, c)     new shape (r,c), same data — r*c MUST equal a.size
  a.reshape(-1)       flatten to 1D — -1 tells NumPy: "you figure out the size"
  a.reshape(-1, 4)    infer rows, fix cols=4 — size must be divisible by 4
  a.reshape(2, -1)    fix rows=2, infer cols

  reshape usually returns a VIEW (same memory) — not a copy.
  Mutating the reshaped result can mutate the original.

FLATTEN vs RAVEL:
  a.ravel()     → 1D VIEW when possible (no copy, fast)
  a.flatten()   → 1D COPY always (safe, independent)

  Use ravel() for read-only scanning.
  Use flatten() when you will modify the result.

SLOW MOTION TRACE on a = np.arange(12):
  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

  reshape(3, 4):          reshape(2, 6):         reshape(4, -1):
  [[ 0  1  2  3]          [[ 0  1  2  3  4  5]   [[ 0  1  2]
   [ 4  5  6  7]           [ 6  7  8  9 10 11]]    [ 3  4  5]
   [ 8  9 10 11]]                                   [ 6  7  8]
                           2*6=12 OK                [ 9 10 11]]
  3*4=12 OK                                        4*3=12 OK

  reshape(-1, 1):
  [[ 0][ 1][ 2]...[ 11]]   shape (12, 1) — column vector

KEY INSIGHT:
  -1 means "I don't want to calculate this dimension — you do it."
  reshape(-1, 1) is the standard trick to turn a 1D array into a column vector
  for broadcasting against a 2D matrix.

TIME / SPACE:
  Time:  O(1) for reshape (just metadata change when it returns a view)
  Space: O(1) for reshape to view / O(n) for flatten (always copies)
```

In [ ]:
import numpy as np

def spiral_order(matrix: list) -> list:
    """
    LC 54 -- Spiral Matrix
    Approach: layer-by-layer peeling using four boundary pointers.
    Args:
        matrix (list[list[int]]): m x n matrix.
    Returns:
        list: all elements in spiral order (1D output regardless of input shape).
    Time:  O(m*n) -- visit every element exactly once
    Space: O(1)   -- output list excluded; only boundary pointers used
    """
    if not matrix:
        return []

    result = []
    top, bottom = 0, len(matrix) - 1       # row boundaries
    left, right = 0, len(matrix[0]) - 1    # column boundaries

    while top <= bottom and left <= right:
        # slow motion on [[1,2,3],[4,5,6],[7,8,9]]:
        # pass1 top=0:   right-> [1,2,3],  down v [6,9],  left<- [8,7],  up ^ [4]
        # pass2 top=1:   boundaries squeeze -> nothing left
        # result: [1,2,3,6,9,8,7,4,5]

        for col in range(left, right + 1):      # walk right along top row
            result.append(matrix[top][col])
        top += 1                                 # top row consumed

        for row in range(top, bottom + 1):       # walk down right column
            result.append(matrix[row][right])
        right -= 1                               # right col consumed

        if top <= bottom:
            for col in range(right, left - 1, -1):  # walk left along bottom row
                result.append(matrix[bottom][col])
            bottom -= 1                          # bottom row consumed

        if left <= right:
            for row in range(bottom, top - 1, -1):  # walk up left column
                result.append(matrix[row][left])
            left += 1                            # left col consumed

    return result


def reshape_demo():
    """Show all reshape variants on the same base array."""
    a = np.arange(12)
    print(f"base:          {a}  shape={a.shape}")
    print(f"reshape(3,4):\n{a.reshape(3,4)}")
    print(f"reshape(-1):   {a.reshape(-1)}    shape={a.reshape(-1).shape}")
    print(f"reshape(-1,1): shape={a.reshape(-1,1).shape}  (column vector)")
    print(f"reshape(2,-1): shape={a.reshape(2,-1).shape}")
    print(f"ravel:         {a.ravel()}  (view, no copy)")
    print(f"flatten:       {a.flatten()}  (always a copy)")


def test_harness(fn):
    tests = [
        ([[1,2,3],[4,5,6],[7,8,9]],           [1,2,3,6,9,8,7,4,5]),
        ([[1,2,3,4],[5,6,7,8],[9,10,11,12]],  [1,2,3,4,8,12,11,10,9,5,6,7]),
        ([[1]],                                [1]),
        ([[1,2],[3,4]],                        [1,2,4,3]),
        ([[1],[2],[3]],                        [1,2,3]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(spiral_order)
reshape_demo()

print("spiral_order defined.")

<a id='8'></a>

## 8. 🧩 Pattern 4: Broadcasting — stretch rules, elementwise ops — LC 238, 560

```
PROBLEM:
  LC 238 — Product of Array Except Self: prefix/suffix arrays added elementwise
  LC 560 — Subarray Sum Equals K: prefix sum array, then check differences

BROADCASTING RULES (3 steps, applied from RIGHT to LEFT across shapes):
  Step 1: Align shapes on the right.
  Step 2: If dimensions match OR one of them is 1 → compatible.
  Step 3: The dimension of size 1 is "stretched" to match the other.

  COMPATIBLE PAIRS (right-aligned):
    (3, 4) + (   4)  →  (4) stretches across all 3 rows  →  result (3, 4)
    (3, 4) + (3, 1)  →  (3,1) stretches across all 4 cols →  result (3, 4)
    (3, 4) + (   1)  →  scalar stretches everywhere         →  result (3, 4)
    (1, 4) + (3, 1)  →  both stretch                        →  result (3, 4)

  INCOMPATIBLE:
    (3, 4) + (3,)    →  right-align: (3,4) vs (3,) → 4 vs 3 → ERROR
    (3, 4) + (2, 4)  →  3 vs 2, neither is 1 → ERROR

SLOW MOTION TRACE:
  a = np.arange(12).reshape(3, 4)     shape (3, 4)
  row_bias = np.array([10, 20, 30])   shape (3,)  <- WRONG -- need (3,1)
  col_bias = np.array([1, 2, 3, 4])   shape (4,)  <- OK -- right-aligned

  a + col_bias:                        a + row_bias.reshape(-1, 1):
  right-align (3,4) vs (4,)            (3,4) vs (3,1)
  last dims 4==4 -> stretch across rows last dims 4 vs 1 -> stretch across cols
  [[ 1  3  5  7]                       [[10 11 12 13]
   [ 5  7  9 11]                        [24 25 26 27]
   [ 9 11 13 15]]                       [38 39 40 41]]

KEY INSIGHT:
  A (n,) vector adds to rows of an (m,n) matrix automatically.
  To add a (m,) vector to COLUMNS you must reshape it to (m,1) first.
  reshape(-1, 1) is the standard fix for "wrong axis broadcast".

TIME / SPACE:
  Time:  O(n) to O(m*n) depending on operation and shapes
  Space: O(1) for the broadcast itself -- NumPy never physically copies the stretched data
```

In [ ]:
import numpy as np
from collections import defaultdict

def subarray_sum(nums: list, k: int) -> int:
    """
    LC 560 -- Subarray Sum Equals K
    Approach: prefix sum + hash map. Count subarrays whose sum equals k.
    Args:
        nums (list[int]): integer array, may contain negatives.
        k (int): target sum.
    Returns:
        int: number of contiguous subarrays with sum == k.
    Time:  O(n) -- single pass
    Space: O(n) -- hash map stores prefix sum counts
    """
    count  = 0
    prefix = 0
    freq   = defaultdict(int)
    freq[0] = 1             # empty prefix has sum 0 -- base case

    for num in nums:
        prefix += num                    # running prefix sum
        # if (prefix - k) was seen before, there is a subarray ending here with sum k
        # slow motion on [1,1,1], k=2:
        # num=1 prefix=1 freq={0:1} look for 1-2=-1 -> not found  freq={0:1,1:1}
        # num=1 prefix=2 freq={0:1,1:1} look for 2-2=0 -> found 1  count=1  freq={0:1,1:1,2:1}
        # num=1 prefix=3 freq={...} look for 3-2=1 -> found 1  count=2  freq={...,3:1}
        # result: 2  (subarrays [1,1] at index 0-1 and 1-2)
        count += freq[prefix - k]
        freq[prefix] += 1

    return count


def broadcasting_demo():
    """Show all broadcasting cases with prints."""
    a = np.arange(12).reshape(3, 4)
    print(f"base a:\n{a}  shape={a.shape}")

    col_bias = np.array([1, 2, 3, 4])          # shape (4,) -- adds to every row
    print(f"a + col_bias (shape {col_bias.shape}):\n{a + col_bias}")

    row_bias = np.array([10, 20, 30])           # shape (3,) -- WRONG for column add
    row_bias_col = row_bias.reshape(-1, 1)       # shape (3,1) -- stretches across cols
    print(f"a + row_bias.reshape(-1,1) (shape {row_bias_col.shape}):\n{a + row_bias_col}")

    scalar = 100                                 # shape () -- stretches everywhere
    print(f"a + 100:\n{a + scalar}")

    # prefix sum demo (LC 238 / 560 pattern)
    nums = np.array([1, 2, 3, 4, 5])
    prefix = np.cumsum(nums)                     # [1, 3, 6, 10, 15]
    suffix = np.cumsum(nums[::-1])[::-1]         # [15, 14, 12, 9, 5]
    print(f"nums   = {nums}")
    print(f"prefix = {prefix}")
    print(f"suffix = {suffix}")


def test_harness(fn):
    tests = [
        ([1, 1, 1],         2, 2),
        ([1, 2, 3],         3, 2),
        ([1, -1, 0],        0, 3),
        ([3, 4, 7, 2, -3, 1, 4, 2], 7, 4),
        ([1],               1, 1),
        ([1],               0, 0),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(subarray_sum)
broadcasting_demo()

print("subarray_sum defined.")

<a id='9'></a>

## 9. 🧩 Pattern 5: 3D Tensors — batch × rows × cols, PyTorch/NumPy parity — Citi HorizonScale

```
REAL-WORLD CONTEXT — CITI HORIZONSCALE ML FORECASTING:
  HorizonScale forecasts telemetry for 6,000 endpoints.
  Each training sample is a time-series window of sensor readings.
  The batch tensor fed to the model has shape (B, T, F):
    B = batch size  (number of samples per gradient step, e.g. 64)
    T = time steps  (lookback window, e.g. 24 hours of readings)
    F = features    (per-timestep metrics: CPU, MEM, NET, etc.)

  Knowing which axis is which is what lets you:
    - Sum across time: np.sum(batch, axis=1) → shape (B, F)
    - Mean per feature: np.mean(batch, axis=2) → shape (B, T)
    - Stack batches:    np.stack([b1, b2], axis=0) → new batch axis

3D TENSOR INDEXING:
  tensor[b]         → full sample b, shape (T, F)
  tensor[b, t]      → timestep t of sample b, shape (F,)
  tensor[b, t, f]   → single scalar value
  tensor[:, :, 0]   → feature 0 across ALL batches and timesteps, shape (B, T)

NUMPY / PYTORCH PARITY:
  NumPy                           PyTorch equivalent
  ─────────────────────────────────────────────────────
  np.zeros((B,T,F))               torch.zeros(B,T,F)
  a.shape                         t.shape
  np.sum(a, axis=1)               t.sum(dim=1)
  np.mean(a, axis=2)              t.mean(dim=2)
  a.reshape(B, -1)                t.reshape(B, -1) or t.view(B, -1)
  a.transpose(0,2,1)              t.permute(0,2,1)
  a[np.newaxis, :]                t.unsqueeze(0)
  a.squeeze()                     t.squeeze()

  The mental model is IDENTICAL — only the method names change.
  If you understand NumPy axis, you understand PyTorch dim.

SLOW MOTION TRACE — mean per endpoint across time:
  batch shape = (4, 3, 2)   (4 samples, 3 timesteps, 2 features)

  np.mean(batch, axis=1):   collapse axis=1 (time) → shape (4, 2)
  for each sample b in 0..3:
    result[b, 0] = mean(batch[b, 0, 0], batch[b, 1, 0], batch[b, 2, 0])
    result[b, 1] = mean(batch[b, 0, 1], batch[b, 1, 1], batch[b, 2, 1])

KEY INSIGHT:
  axis=1 (the time axis) is "eaten" — you get one average per (sample, feature) pair.
  This is how you compute "average CPU usage per endpoint over the last 24h."

TIME / SPACE:
  Time:  O(B*T*F) — visit every element
  Space: O(B*F)   — output shape after collapsing time axis
```

In [ ]:
import numpy as np

def simulate_horizonscale_batch(B=4, T=6, F=3, seed=42):
    """
    Simulate a HorizonScale-style training batch.
    B = batch size  (number of endpoint time-series samples)
    T = time steps  (lookback window -- e.g. 6 hours of readings)
    F = features    (per-timestep: CPU_pct, MEM_pct, NET_mbps)
    Returns ndarray of shape (B, T, F).
    """
    rng = np.random.default_rng(seed)
    # each endpoint has different baseline -- add noise
    batch = rng.uniform(0, 100, size=(B, T, F)).round(1)
    return batch


def analyze_batch(batch):
    """Run the axis operations you actually use in HorizonScale preprocessing."""
    B, T, F = batch.shape
    print(f"batch shape : {batch.shape}  (B={B} samples, T={T} timesteps, F={F} features)")
    print(f"batch[0]    shape={batch[0].shape}   (all timesteps for sample 0)")
    print(f"batch[0,0]  shape={batch[0,0].shape}  (all features at t=0 for sample 0)")
    print(f"batch[0,0,0] = {batch[0,0,0]}          (scalar -- CPU_pct at t=0, sample 0)")

    # mean per feature across ALL time steps and ALL samples
    global_mean = np.mean(batch, axis=(0, 1))   # collapse B and T -> shape (F,)
    print(f"\nmean per feature (axis=(0,1)): {global_mean}  shape={global_mean.shape}")

    # mean per timestep per sample (collapse features)
    sample_time_mean = np.mean(batch, axis=2)   # collapse F -> shape (B, T)
    print(f"mean per (sample,timestep) (axis=2): shape={sample_time_mean.shape}")

    # mean per sample across time (collapse time axis)
    sample_mean = np.mean(batch, axis=1)        # collapse T -> shape (B, F)
    print(f"mean per (sample,feature) (axis=1): shape={sample_mean.shape}")
    print(f"sample averages:\n{sample_mean.round(1)}")

    # flatten each sample to 1D for a simple dense layer
    flat = batch.reshape(B, -1)                 # (B, T*F)
    print(f"\nflattened for dense layer: shape={flat.shape}  (B={B}, T*F={T*F})")

    # stack two batches along new batch axis
    batch2 = batch * 0.9                        # simulate a second batch
    combined = np.concatenate([batch, batch2], axis=0)  # (2B, T, F)
    print(f"concatenated batches: shape={combined.shape}")


def pytorch_parity_note():
    """Print the NumPy -> PyTorch translation table."""
    print("NumPy  ->  PyTorch equivalents:")
    print("  a.shape              ->  t.shape")
    print("  np.sum(a, axis=1)    ->  t.sum(dim=1)")
    print("  np.mean(a, axis=2)   ->  t.mean(dim=2)")
    print("  a.reshape(B, -1)     ->  t.reshape(B, -1)  or  t.view(B, -1)")
    print("  a.transpose(0,2,1)   ->  t.permute(0,2,1)")
    print("  a[np.newaxis,:]      ->  t.unsqueeze(0)")
    print("  a.squeeze()          ->  t.squeeze()")


batch = simulate_horizonscale_batch(B=4, T=6, F=3)
analyze_batch(batch)
pytorch_parity_note()

print("\nsimulate_horizonscale_batch defined.")

<a id='10'></a>

## 10. The Multidimensional Arrays Decision Map

```
QUESTION TYPE                              KEY TECHNIQUE                  LC / REAL-WORLD
────────────────────────────────────────────────────────────────────────────────────────
search sorted 2D matrix (both axes sorted) staircase from top-right        LC 240
search sorted 2D matrix (rows+col sorted)  binary search with // and %     LC 74
rotate matrix 90 degrees clockwise         transpose + fliplr / row.rev    LC 48
collect elements in spiral order           4-pointer boundary peeling      LC 54
sum/mean per row (collapse cols)           np.sum(a, axis=1)               --
sum/mean per column (collapse rows)        np.sum(a, axis=0)               --
flatten 2D for 1D algorithm                a.ravel() view / a.flatten()    LC 74, 240
reshape without knowing one dimension      a.reshape(-1, k) -- -1 inferred  LC 48, 54
add row vector to every row of matrix      a + vec  (broadcasts if shape (n,))
add col vector to every col of matrix      a + vec.reshape(-1,1) -- must be (m,1)
prefix/suffix sums, elementwise product    np.cumsum, a * b elementwise    LC 238, 560
3D batch tensor: mean across time          np.mean(batch, axis=1)          Citi HorizonScale
3D batch tensor: mean across features      np.mean(batch, axis=2)          Citi HorizonScale
flatten each sample in a batch             batch.reshape(B, -1)            ML preprocessing
stack two batches                          np.concatenate([b1,b2], axis=0) ML preprocessing
```

<a id='11'></a>

## 11. Interview Cheat Sheet

### 1. When to reach for NumPy ndarray

| Signal | What to Do |
|--------|------------|
| Need matrix dimensions | `a.shape`, `a.ndim`, `a.size` |
| Matrix sorted in rows and columns | Staircase search from top-right corner |
| Rotate matrix in-place | Transpose + flip left-right |
| Spiral traversal | 4-boundary-pointer peel loop |
| Sum or mean per row | `np.sum(a, axis=1)` — axis 1 collapses cols |
| Sum or mean per column | `np.sum(a, axis=0)` — axis 0 collapses rows |
| Add vector to every row | `a + vec` where vec has shape `(n,)` |
| Add vector to every column | `a + vec.reshape(-1, 1)` — must be `(m,1)` |
| Flatten for 1D scan | `a.ravel()` (view) or `a.flatten()` (copy) |
| Change shape, keep data | `a.reshape(r, c)` — verify `r*c == a.size` |
| 3D batch tensor | shape `(B, T, F)` — `axis=1` collapses time |

---

### 2. The O(1) operations — memorize these

```python
a.shape        # tuple of sizes -- O(1)
a.ndim         # number of axes -- O(1)
a.size         # total elements -- O(1)
a.dtype        # element type   -- O(1)
a[i, j]        # element access -- O(1)
a[i]           # row view       -- O(1), no copy
a[:, j]        # col view       -- O(1), no copy
a.reshape(r,c) # metadata only  -- O(1) when returns view
a.ravel()      # flatten view   -- O(1) when contiguous
a.T            # transpose view -- O(1), no copy
```

---

### 3. Common templates

```python
# STAIRCASE SEARCH -- LC 240
def staircase_search(matrix, target):
    m, n = len(matrix), len(matrix[0])
    row, col = 0, n - 1          # start top-right
    while row < m and col >= 0:
        val = matrix[row][col]
        if val == target:   return True
        elif val > target:  col -= 1   # eliminate column
        else:               row += 1   # eliminate row
    return False

# TRANSPOSE + FLIPLR ROTATION -- LC 48
def rotate_cw(matrix):
    import numpy as np
    a = np.array(matrix)
    return np.fliplr(a.T).tolist()   # transpose then flip left-right

# SPIRAL COLLECT -- LC 54
def spiral(matrix):
    result = []
    top, bottom, left, right = 0, len(matrix)-1, 0, len(matrix[0])-1
    while top <= bottom and left <= right:
        for c in range(left, right+1):       result.append(matrix[top][c])
        top += 1
        for r in range(top, bottom+1):       result.append(matrix[r][right])
        right -= 1
        if top <= bottom:
            for c in range(right, left-1,-1): result.append(matrix[bottom][c])
            bottom -= 1
        if left <= right:
            for r in range(bottom, top-1,-1): result.append(matrix[r][left])
            left += 1
    return result

# PREFIX SUM + HASH MAP -- LC 560
def subarray_sum_k(nums, k):
    from collections import defaultdict
    freq = defaultdict(int)
    freq[0] = 1
    prefix = count = 0
    for n in nums:
        prefix += n
        count  += freq[prefix - k]
        freq[prefix] += 1
    return count

# 3D BATCH TENSOR AXIS OPS -- HorizonScale
import numpy as np
batch = np.zeros((64, 24, 6))      # B=64 endpoints, T=24h, F=6 features
time_mean   = np.mean(batch, axis=1)   # collapse time  -> (64, 6)
feat_mean   = np.mean(batch, axis=2)   # collapse feats -> (64, 24)
flat_batch  = batch.reshape(64, -1)    # flatten for dense layer -> (64, 144)
```

---

### 4. Gotchas

```
X  np.sum(a)                     no axis -- collapses to scalar -- usually wrong
X  a + row_vec where row_vec shape (m,)  wrong axis -- need reshape(-1,1)
X  reshape without size check    r*c must equal a.size or ValueError
X  assume reshape copies         it usually returns a view -- mutating it mutates original
X  a[0] for 2D mutates original  row views are NOT copies
OK a.reshape(-1,1)               always the fix for "wrong axis broadcast"
OK a.ravel() for read-only scan  no copy, O(1) overhead
OK a.flatten() when you need independent copy
OK axis=0 eats rows -> result shape loses first dim
OK axis=1 eats cols -> result shape loses second dim
OK PyTorch dim= is exactly NumPy axis= -- same rules, different name
```

```
              🧩 MULTIDIMENSIONAL ARRAYS & TENSORS -- MASTER MAP

                         ndarray (contiguous memory)
                                    |
          .----------------.--------+--------.----------------.
          |                |                 |                |
       SHAPE            AXIS             RESHAPE          BROADCAST
       .shape           axis=0 eats rows  reshape(r,c)    right-align shapes
       .ndim            axis=1 eats cols  reshape(-1)     dim of 1 stretches
       .size            axis=(0,1) both   reshape(-1,1)   (n,) adds to rows
       .dtype           result loses      ravel() view    (m,1) adds to cols
                        that axis         flatten() copy
          |                |                 |                |
       LC 74           LC 48 rotate      LC 54 spiral    LC 238 prefix
       LC 240          LC 54 spiral      LC 48 rotate    LC 560 subarray
       staircase       transpose+fliplr  boundary peel   np.cumsum
          |
       3D TENSORS -- shape (B, T, F)
       axis=0 -> across batch
       axis=1 -> across time     <- np.mean(batch, axis=1) = avg CPU over time
       axis=2 -> across features
       reshape(B,-1) -> flatten for dense layer
       PyTorch: dim= is exactly axis=
       Citi HorizonScale: batch=(64,24,6) -> B=64 endpoints, T=24h, F=6 metrics
```

---
*End of Multidimensional Arrays and Tensors Master Guide — Sean Edition*